# Twitch Emote Prediction — Embedding Extraction

Extracts joint audio-visual embeddings from a scraped Twitch VOD dataset produced by the Twitch VOD Scraper notebook. Each output is a `.npz` file per clip containing two token sequences:

- **Video**: encoded by V-JEPA 2 (`facebook/vjepa2-vitl-fpc64-256`) — spatial mean-pool per temporal step → `[32, 1024]`
- **Audio**: encoded by Whisper (`openai/whisper-small`, encoder only) — stride-sampled over the 15s active window → `[125, 768]`

### Sections

- **Configuration**: Set `ZIP_FILE_PATH` to the dataset zip on Google Drive before running.
- **Setup**: Mounts Drive, unzips the dataset.
- **V-JEPA 2**: Extracts per-clip video token sequences; skips already-processed clips.
- **Whisper**: Extracts per-clip audio token sequences; skips already-processed clips.
- **Concatenate**: Packages video and audio tokens into a single `.npz` file per clip.
- **Export**: Zips combined embeddings and saves to the same Drive folder as the input.
- **Health Check**: Validates all embeddings for correct shape, NaN, Inf, and near-zero variance.

### Prerequisites

No Colab secrets required. Input must be a zip produced by the Twitch VOD Scraper, structured as:

| Path | Contents |
|---|---|
| `dataset_{vod_id}/video/` | 64-frame MP4 clips (256×256) |
| `dataset_{vod_id}/audio/` | corresponding MP3 clips (~15s) |

## Imports

In [ ]:
import torch
import numpy as np
import cv2
import random
import shutil
import zipfile
import time
from pathlib import Path
from tqdm import tqdm
from transformers import AutoVideoProcessor, AutoModel, WhisperModel, AutoFeatureExtractor
from torch.utils.data import Dataset, DataLoader
from accelerate import Accelerator
import librosa
import torchaudio.transforms as T


## Configuration

In [ ]:
# ── Input ──────────────────────────────────────────────────────────────────────
ZIP_FILE_PATH = '/content/drive/MyDrive/twitch_emote_data/dataset_1594760722_clips.zip'  # ← your dataset zip in Drive

# ── V-JEPA 2 ───────────────────────────────────────────────────────────────────
VJEPA_HF_REPO     = "facebook/vjepa2-vitl-fpc64-256"
VJEPA_BATCH_SIZE  = 4
VJEPA_NUM_WORKERS = 4
# vjepa2-vitl-fpc64-256: 64 frames / tubelet_size 2 = 32 temporal steps, (256/16)^2 = 256 spatial patches
VJEPA_TEMPORAL_TOKENS = 32
VJEPA_SPATIAL_TOKENS  = 256

# ── Whisper ─────────────────────────────────────────────────────────────────────
WHISPER_HF_REPO     = "openai/whisper-small"
WHISPER_BATCH_SIZE  = 32
WHISPER_NUM_WORKERS = 0      # 0 avoids multiprocessing crashes in Colab
TARGET_SR             = 16000
WHISPER_SAMPLE_LENGTH = 480000  # 30s * 16kHz — Whisper's full context window
WHISPER_ACTIVE_TOKENS = 750    # tokens corresponding to the 15s clip in a 30s window
WHISPER_AUDIO_TOKENS  = 125    # tokens kept after stride-6 sampling

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
accelerator = Accelerator()
device = accelerator.device

def log(msg):
    if accelerator.is_main_process:
        print(msg)

output_dir = Path('./')

dataset_folder_name = Path(ZIP_FILE_PATH).name.replace("_clips.zip", "")
dataset_folder = output_dir / dataset_folder_name
required_subdirs = ["video", "audio", "target"]

already_extracted = all(
    (dataset_folder / sub).exists() and any((dataset_folder / sub).iterdir())
    for sub in required_subdirs
)

if already_extracted:
    log(f"Dataset already extracted to '{dataset_folder_name}', skipping unzip.")
elif Path(ZIP_FILE_PATH).exists():
    log(f"Unzipping {ZIP_FILE_PATH} to {output_dir}...")
    with zipfile.ZipFile(ZIP_FILE_PATH, 'r') as zip_ref:
        zip_ref.extractall(output_dir)
    log("Unzipping complete.")
else:
    log(f"⚠️ Zip file not found at {ZIP_FILE_PATH}.")

log(f"Running on {accelerator.num_processes} GPU(s) ({device})")

## V-JEPA 2

In [ ]:
class VideoClipDataset(Dataset):
    def __init__(self, video_dir, embed_dir):
        self.video_dir = Path(video_dir)
        self.embed_dir = Path(embed_dir)
        self.embed_dir.mkdir(parents=True, exist_ok=True)

        all_videos = sorted(self.video_dir.glob("*.mp4"))
        self.video_files = [
                v for v in all_videos
                if not (self.embed_dir / f"{v.stem}.npy").exists()
        ]

    def __len__(self):
        return len(self.video_files)

    def __getitem__(self, idx):
        video_path = self.video_files[idx]
        clip_id = video_path.stem

        cap = cv2.VideoCapture(str(video_path))
        frames = []

        while True:
            ret, frame = cap.read()
            if not ret:
                break
            frame_rgb = cv2.cvtColor(frame, cv2.COLOR_BGR2RGB)
            frames.append(frame_rgb)
        cap.release()

        if len(frames) == 0:
            frames = [np.zeros((256, 256, 3), dtype=np.uint8)] * 64

        video = np.stack(frames)
        return {"video": video, "clip_id": clip_id}


def collate_fn(batch, processor):
    videos = [item["video"] for item in batch]
    clip_ids = [item["clip_id"] for item in batch]

    inputs = processor(videos, return_tensors="pt")

    return {
            "pixel_values": inputs["pixel_values_videos"],
            "clip_ids": clip_ids
    }


def extract_embeddings_distributed(dataset_folder):
    dataset_folder = Path(dataset_folder)
    video_dir = dataset_folder / "video"

    dataset_id = dataset_folder.name.split('_')[-1]
    embed_dir = Path(f"./jepa-embeddings_{dataset_id}")

    log("Loading V-JEPA 2")

    max_retries = 3
    for attempt in range(max_retries):
            try:
                    force = attempt > 0

                    if force:
                            log(f"⚠️ Corrupted weights detected. Forcing fresh redownload (Attempt {attempt + 1})...")

                    model = AutoModel.from_pretrained(VJEPA_HF_REPO, force_download=force)
                    processor = AutoVideoProcessor.from_pretrained(VJEPA_HF_REPO, force_download=force)

                    log("Model and processor loaded successfully.")
                    break

            except Exception as e:
                    log(f"❌ Attempt {attempt + 1} failed to load model: {e}")
                    if attempt == max_retries - 1:
                            log("❌ Critical error: could not load model after multiple attempts. Check disk space.")
                            raise e
                    time.sleep(5)

    model.eval()

    dataset = VideoClipDataset(video_dir, embed_dir)
    log(f"Found {len(dataset)} unprocessed clips")

    if len(dataset) == 0:
        log("All clips already processed.")
        return

    dataloader = DataLoader(
            dataset,
            batch_size=VJEPA_BATCH_SIZE,
            shuffle=False,
            num_workers=VJEPA_NUM_WORKERS,
            collate_fn=lambda b: collate_fn(b, processor),
            pin_memory=True
    )

    model, dataloader = accelerator.prepare(model, dataloader)

    processed_count = 0

    progress_bar = tqdm(
        dataloader,
        desc=f"GPU {accelerator.process_index}",
        disable=not accelerator.is_local_main_process
    )

    for batch in progress_bar:
        pixel_values = batch["pixel_values"]
        clip_ids = batch["clip_ids"]

        with torch.no_grad():
            outputs = model(pixel_values_videos=pixel_values)
            # [batch, 8192, 1024] → [batch, 32, 256, 1024] → mean over spatial → [batch, 32, 1024]
            hidden = outputs.last_hidden_state
            hidden = hidden.view(hidden.shape[0], VJEPA_TEMPORAL_TOKENS, VJEPA_SPATIAL_TOKENS, hidden.shape[-1])
            embeddings = hidden.mean(dim=2)

        embeddings_np = embeddings.cpu().numpy()
        for clip_id, emb in zip(clip_ids, embeddings_np):
            embed_path = embed_dir / f"{clip_id}.npy"
            np.save(embed_path, emb)
            processed_count += 1

    accelerator.wait_for_everyone()
    total_processed = torch.tensor([processed_count], device=device)
    total_processed = accelerator.gather(total_processed).sum().item()

    log(f"Processed {int(total_processed)} clips")

### Execution

In [ ]:
dataset_folders = sorted(Path(".").glob("dataset_*"))
log(f"Found {len(dataset_folders)} dataset folders")

for folder in dataset_folders:
    extract_embeddings_distributed(folder)

log("All embeddings extracted.")


## Whisper

In [ ]:
class AudioClipDataset(Dataset):
    def __init__(self, audio_dir, embed_dir):
        self.audio_dir = Path(audio_dir)
        self.embed_dir = Path(embed_dir)
        self.embed_dir.mkdir(parents=True, exist_ok=True)

        all_audio = sorted(self.audio_dir.glob("*.mp3"))

        self.audio_files = [
            f for f in all_audio
            if not (self.embed_dir / f"{f.stem}.npy").exists()
        ]

    def __len__(self):
        return len(self.audio_files)

    def __getitem__(self, idx):
        audio_path = self.audio_files[idx]
        clip_id = audio_path.stem

        try:
            audio_array, sr = librosa.load(str(audio_path), sr=None, mono=False)
        except Exception as e:
            log(f"⚠️ Error loading {audio_path}: {e}")
            return {"audio": np.zeros(WHISPER_SAMPLE_LENGTH), "clip_id": clip_id}

        waveform = torch.from_numpy(audio_array)

        if waveform.ndim == 1:
            waveform = waveform.unsqueeze(0)

        if sr != TARGET_SR:
            resampler = T.Resample(sr, TARGET_SR)
            waveform = resampler(waveform)

        if waveform.shape[0] > 1:
            waveform = torch.mean(waveform, dim=0, keepdim=True)

        audio_array = waveform.squeeze().numpy()

        return {"audio": audio_array, "clip_id": clip_id}


def collate_fn_audio(batch, processor):
    raw_audio = [item["audio"] for item in batch]
    clip_ids = [item["clip_id"] for item in batch]

    padded_audio = []

    for audio in raw_audio:
        curr_len = len(audio)
        if curr_len < WHISPER_SAMPLE_LENGTH:
            pad_len = WHISPER_SAMPLE_LENGTH - curr_len
            audio = np.pad(audio, (0, pad_len), 'constant')
        else:
            audio = audio[:WHISPER_SAMPLE_LENGTH]
        padded_audio.append(audio)

    inputs = processor(
        padded_audio,
        sampling_rate=TARGET_SR,
        return_tensors="pt"
    )

    return {
        "input_features": inputs.input_features,
        "clip_ids": clip_ids
    }


def extract_audio_embeddings_distributed(dataset_folder):
    dataset_folder = Path(dataset_folder)
    audio_dir = dataset_folder / "audio"

    dataset_id = dataset_folder.name.split('_')[-1]
    embed_dir = Path(f"./whisper-embeddings_{dataset_id}")

    log(f"Loading Whisper: {WHISPER_HF_REPO}")

    max_retries = 3
    for attempt in range(max_retries):
        try:
            force = attempt > 0
            if force:
                log(f"⚠️ Corrupted weights detected. Forcing fresh redownload (Attempt {attempt + 1})...")
            model = WhisperModel.from_pretrained(WHISPER_HF_REPO, force_download=force)
            processor = AutoFeatureExtractor.from_pretrained(WHISPER_HF_REPO, force_download=force)
            del model.decoder
            log("Model and processor loaded successfully.")
            break
        except Exception as e:
            log(f"❌ Attempt {attempt + 1} failed to load model: {e}")
            if attempt == max_retries - 1:
                log("❌ Critical error: could not load model after multiple attempts. Check disk space.")
                raise e
            time.sleep(5)

    model.eval()

    dataset = AudioClipDataset(audio_dir, embed_dir)
    log(f"Found {len(dataset)} unprocessed audio clips")

    if len(dataset) == 0:
        log("All clips already processed.")
        return

    dataloader = DataLoader(
        dataset,
        batch_size=WHISPER_BATCH_SIZE,
        shuffle=False,
        num_workers=WHISPER_NUM_WORKERS,
        collate_fn=lambda b: collate_fn_audio(b, processor),
        pin_memory=True
    )

    model, dataloader = accelerator.prepare(model, dataloader)

    processed_count = 0
    stride = WHISPER_ACTIVE_TOKENS // WHISPER_AUDIO_TOKENS

    progress_bar = tqdm(
        dataloader,
        desc=f"GPU {accelerator.process_index}",
        disable=not accelerator.is_local_main_process
    )

    for batch in progress_bar:
        input_features = batch["input_features"]
        clip_ids = batch["clip_ids"]

        with torch.no_grad():
            encoder_outputs = model.encoder(input_features)
            # [batch, 1500, 768] → trim to active 15s window → stride sample → [batch, 125, 768]
            embeddings = encoder_outputs.last_hidden_state[:, :WHISPER_ACTIVE_TOKENS:stride, :]

        embeddings_np = embeddings.cpu().numpy()

        for clip_id, emb in zip(clip_ids, embeddings_np):
            embed_path = embed_dir / f"{clip_id}.npy"
            np.save(embed_path, emb)
            processed_count += 1

    accelerator.wait_for_everyone()
    total_processed = torch.tensor([processed_count], device=device)
    total_processed = accelerator.gather(total_processed).sum().item()
    log(f"Processed {int(total_processed)} clips in {dataset_folder.name}")

### Execution

In [ ]:
dataset_folders = sorted(Path(".").glob("dataset_*"))
log(f"Found {len(dataset_folders)} dataset folders")

for folder in dataset_folders:
    extract_audio_embeddings_distributed(folder)

log("All audio embeddings extracted.")


## Concatenate

In [ ]:
dataset_folders = sorted(Path(".").glob("dataset_*"))

def concatenate_embeddings(dataset_folder):
    dataset_folder = Path(dataset_folder)
    dataset_id = dataset_folder.name.split('_')[-1]

    audio_embed_dir = Path(f"./whisper-embeddings_{dataset_id}")
    video_embed_dir = Path(f"./jepa-embeddings_{dataset_id}")
    combined_embed_dir = Path(f"./embeddings_{dataset_id}")

    combined_embed_dir.mkdir(parents=True, exist_ok=True)

    if not audio_embed_dir.exists():
        log(f"⚠️ Audio embeddings directory not found: {audio_embed_dir}")
        return

    if not video_embed_dir.exists():
        log(f"⚠️ Video embeddings directory not found: {video_embed_dir}")
        return

    audio_files = sorted(audio_embed_dir.glob("*.npy"))

    if len(audio_files) == 0:
        log(f"⚠️ No audio embeddings found in {audio_embed_dir}")
        return

    successful = 0
    missing_video = 0

    log(f"Processing {len(audio_files)} clips from {dataset_folder.name}")

    for audio_file in tqdm(audio_files, desc=f"Concatenating {dataset_folder.name}"):
        clip_id = audio_file.stem
        video_file = video_embed_dir / f"{clip_id}.npy"
        combined_file = combined_embed_dir / f"{clip_id}_embeddings.npz"

        if combined_file.exists():
            continue

        if not video_file.exists():
            missing_video += 1
            continue

        try:
            audio_emb = np.load(audio_file)  # [125, 768]
            video_emb = np.load(video_file)  # [32, 1024]

            np.savez(combined_file, video=video_emb, audio=audio_emb)
            successful += 1

        except Exception as e:
            log(f"⚠️ Error processing {clip_id}: {e}")
            continue

    log(f"Finished {dataset_folder.name}:")
    log(f"  - Successfully combined: {successful}")
    log(f"  - Missing video embeddings: {missing_video}")
    log(f"  - Combined embeddings saved to: {combined_embed_dir}")

for folder in dataset_folders:
    concatenate_embeddings(folder)

log("All embeddings concatenated.")

## Export

In [ ]:
drive_output_folder = Path(ZIP_FILE_PATH).parent

for folder in dataset_folders:
    dataset_id = folder.name.split('_')[-1]
    embed_dir = Path(f"./embeddings_{dataset_id}")

    if not embed_dir.exists():
        log(f"⚠️ Embedding directory '{embed_dir.name}' not found, skipping.")
        continue

    zip_path = Path(f"embeddings_{dataset_id}.zip")
    log(f"Compressing '{embed_dir.name}'...")
    shutil.make_archive(str(zip_path.stem), 'zip', '.', embed_dir)

    dest = drive_output_folder / zip_path.name
    shutil.copy(zip_path, dest)
    zip_path.unlink()
    log(f"Saved to Drive: {dest}")

## Health Check

In [ ]:
for folder in dataset_folders:
    dataset_id = folder.name.split('_')[-1]
    embed_dir = Path(f"./embeddings_{dataset_id}")

    if not embed_dir.exists():
        log(f"⚠️ Embedding directory '{embed_dir.name}' not found, skipping.")
        continue

    npz_files = list(embed_dir.glob("*.npz"))
    if not npz_files:
        log(f"⚠️ No embedding files found in {embed_dir.name}")
        continue

    total = len(npz_files)
    issues = 0

    for file in npz_files:
        data = np.load(file)
        video = data['video']  # expected [32, 1024]
        audio = data['audio']  # expected [125, 768]

        has_issue = (
            video.shape != (VJEPA_TEMPORAL_TOKENS, 1024) or
            audio.shape != (WHISPER_AUDIO_TOKENS, 768) or
            np.isnan(video).any() or np.isinf(video).any() or np.std(video) < 1e-6 or
            np.isnan(audio).any() or np.isinf(audio).any() or np.std(audio) < 1e-6
        )
        if has_issue:
            issues += 1

    ok = total - issues
    log(f"  {embed_dir.name}: {total} embeddings — {ok} ok ({100*ok/total:.1f}%), {issues} flagged ({100*issues/total:.1f}%)")